# Session 2: the ledger behind a payment
**A. Real-world problem.** You tap "pay", the screen freezes, so you tap again. Did the shop
get paid once or twice? A bank must answer that correctly millions of times a day. The tools
are old and simple: a *double-entry ledger* (every payment is written twice, once as money
leaving and once as money arriving), *atomic* transactions (both writes happen or neither),
and *idempotency* (doing the same request again changes nothing the second time).

**Plan.** (1) build the ledger, (2) make a payment all-or-nothing, (3) make retries safe,
(4) decide in which order a bank should check things before it debits.

General education about how such systems are built; not advice about real accounts.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: a ledger of entries
An *entry* is (transaction id, account, signed amount). Money leaving an account is negative,
money arriving is positive (for a customer's account, a bank calls the first a *debit* and the
second a *credit*). A balance is never stored on its own here: it is the **sum of the
account's entries**. A special account `MINT` is where money enters the toy world, so that
every transaction, even the first deposit, has two sides. Two facts follow:

    every transaction:  sum of its entries = 0
    whole ledger:       sum of all entries = 0, so customer money = money deposited

In [ ]:
class Ledger:
    def __init__(self):
        self.entries = []         # (txn_id, account, signed amount)
        self.results = {}         # txn_id -> first reply (this is the idempotency memory)

    def deposit(self, account, amount):
        self.entries += [(f"dep-{len(self.entries)}", "MINT", -amount), (f"dep-{len(self.entries)}", account, amount)]

    def balance(self, account):
        return sum(a for _, acct, a in self.entries if acct == account)

    def customer_total(self):
        return sum(a for _, acct, a in self.entries if acct != "MINT")

    def ledger_sum(self):
        return sum(a for _, _, a in self.entries)

## Step 2: atomic or not?
Suppose the program crashes between writing the debit and writing the credit. Compare two ways
of writing a transfer. The careless one writes line by line. The atomic one first prepares both
lines and then writes them together, so a crash before the write leaves nothing behind.

In [ ]:
class Crash(Exception):
    """Stands in for a power cut or a software fault at the worst moment."""


def transfer_careless(ledger, txn_id, src, dst, amount, crash=False):
    ledger.entries.append((txn_id, src, -amount))
    if crash:
        raise Crash()
    ledger.entries.append((txn_id, dst, amount))


def transfer_atomic(ledger, txn_id, src, dst, amount, crash=False):
    prepared = [(txn_id, src, -amount), (txn_id, dst, amount)]
    if crash:
        raise Crash()
    ledger.entries.extend(prepared)           # one step: both lines or none

## Step 3: what to check before debiting, and in which order
A request carries: a transaction id, who pays, who is paid, an amount, and proof that the owner
approved it (here a simple True/False `authenticated`; Session 3 checks a real PIN hash).

1. **Authenticated?** First, so strangers learn nothing about accounts.
2. **Duplicate id?** If this id was seen, return the *same answer as before*. This check must
   come BEFORE the balance check: after a successful payment of 900 from 1000, the balance is
   100, so a retry would fail the balance test and wrongly say "declined" for a payment that
   actually happened.
3. **Valid amount?** Positive whole rupees, and not more than the per-payment limit.
4. **Different accounts?**
5. **Sufficient balance?**
Only then commit atomically.

In [ ]:
def pay(ledger, txn_id, src, dst, amount, authenticated=True, per_payment_limit=2000, dup_first=True):
    """Returns a reply dict: status 'ok' or 'declined', a reason, and whether it is a replay."""

    def reply(status, reason=""):
        r = {"status": status, "reason": reason, "replayed": False}
        if reason != "not authenticated":
            ledger.results[txn_id] = r
        return r

    if not authenticated:
        return reply("declined", "not authenticated")
    if dup_first and txn_id in ledger.results:
        return {**ledger.results[txn_id], "replayed": True}
    if not isinstance(amount, int) or amount <= 0:
        return reply("declined", "bad amount")
    if amount > per_payment_limit:
        return reply("declined", "over per-payment limit")
    if src == dst:
        return reply("declined", "same account")
    if ledger.balance(src) < amount:
        return reply("declined", "insufficient balance")
    if not dup_first and txn_id in ledger.results:       # the wrong order, for comparison
        return {**ledger.results[txn_id], "replayed": True}
    transfer_atomic(ledger, txn_id, src, dst, amount)
    return reply("ok")

## Step 4: retries with and without a transaction id check
A phone whose reply was lost resends the request `retries` times. `charge_count` is how many
times the payer was debited. With the id check it is 1 (or 0 if declined). Without it, every
arrival is a new payment until the balance runs out.

In [ ]:
def charge_count(amount, retries, balance=1000, idempotent=True):
    if amount <= 0 or amount > balance:
        return 0
    if idempotent:
        return 1
    return min(retries, balance // amount)


def payer_after(amount, retries, balance=1000, idempotent=True):
    return balance - amount * charge_count(amount, retries, balance, idempotent)


def retry_demo(amount, retries, idempotent):
    """The same idea run through the real Ledger: returns (payer balance, payee balance)."""
    led = Ledger()
    led.deposit("payer", 1000)
    led.deposit("payee", 400)
    for k in range(retries):
        tid = "T1" if idempotent else f"T1-attempt{k}"      # a new id per attempt = no id check
        pay(led, tid, "payer", "payee", amount)
    return led.balance("payer"), led.balance("payee")

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    led = Ledger()
    led.deposit("payer", 1000)
    led.deposit("payee", 400)
    print("start: payer", led.balance("payer"), "payee", led.balance("payee"), "customer total", led.customer_total())
    print("T1 pay 250     ->", pay(led, "T1", "payer", "payee", 250))
    print("T1 again       ->", pay(led, "T1", "payer", "payee", 250))
    print("T2 pay 900     ->", pay(led, "T2", "payer", "payee", 900))
    print("end:   payer", led.balance("payer"), "payee", led.balance("payee"), "customer total", led.customer_total(),
          "ledger sum", led.ledger_sum())
    print("entries:")
    for e in led.entries:
        print("  ", e)

    for fn in (transfer_careless, transfer_atomic):
        t = Ledger()
        t.deposit("payer", 1000)
        t.deposit("payee", 400)
        try:
            fn(t, "X", "payer", "payee", 250, crash=True)
        except Crash:
            pass
        print(f"{fn.__name__:17s} after a crash: payer {t.balance('payer')} payee {t.balance('payee')} "
              f"customer total {t.customer_total()} ledger sum {t.ledger_sum()}")

    bad = Ledger()
    bad.deposit("payer", 1000)
    bad.deposit("payee", 400)
    pay(bad, "T1", "payer", "payee", 900)
    print("retry of a 900 payment, check order duplicate-first :", pay(bad, "T1", "payer", "payee", 900))
    print("retry of a 900 payment, check order balance-first   :", pay(bad, "T1", "payer", "payee", 900, dup_first=False))
    print("3 retries of 250, no id check:", retry_demo(250, 3, False), " with id check:", retry_demo(250, 3, True))

    # figure 1: ledger lines and balances after the three requests
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    names = ["payer", "payee"]
    ax[0].bar(names, [1000, 400], color="#9aa5b1", label="before")
    ax[0].bar(names, [led.balance(n) for n in names], width=0.4, color="#2a9d8f", label="after T1 (twice) and T2")
    ax[0].set_ylabel("balance (rupees)")
    ax[0].set_title("Balances are sums of entries")
    ax[0].legend(fontsize=8)
    ax[0].grid(alpha=0.3, axis="y")
    cum = np.cumsum([a for _, acct, a in led.entries if acct in names])
    ax[1].step(range(len(cum)), cum, where="post")
    ax[1].set_xlabel("entry number (customer entries only)")
    ax[1].set_ylabel("running customer total (rupees)")
    ax[1].set_title("Customer money only changes when money is deposited")
    ax[1].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "ledger_balances.png", dpi=120)

    # figure 2: charges versus retries
    r = np.arange(1, 9)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(r, [payer_after(250, k, idempotent=False) for k in r], "o-", label="no id check")
    ax2.plot(r, [payer_after(250, k, idempotent=True) for k in r], "s--", label="with id check")
    ax2.set_xlabel("number of times the request arrives (count)")
    ax2.set_ylabel("payer balance after (rupees)")
    ax2.set_title("Paying 250 from 1000: retries")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "retries.png", dpi=120)

## Where this stops
A real bank has many ledgers, not one list in memory. Session 3 turns this into a small
switch that you can stress with lost messages and fraud rules.

## Coding tasks
1. **Easy.** Run `python start.py run session-02/lesson.py` and read the printed entries. Predict by hand the balances after one more payment T3 of 100 from the payee to the payer, then check.
2. **Stretch.** Change `pay` so that it also refuses a payment of more than 50 percent of the payer's balance. Which place in the order of checks is best, and why?
3. **Explain.** Explain why the whole ledger sums to zero even though customers' balances are positive. Which account is negative?

## Assessment
- **A. Conceptual.** Why is it a problem if a program writes the debit, then crashes before writing the credit? What does writing both entries in one step change?
- **B. Numerical.** The payer has Rs 1000. A request for Rs 300 arrives 5 times. How much is the payer charged in total, and what are both balances, (a) without an id check and (b) with one? The payee starts with Rs 400.
- **C. Graph and simulation interpretation.** In the retries figure (Rs 250 from Rs 1000), what are the payer balances without the id check for 1, 2, 3, 4 and 8 arrivals, and why does the red curve go flat?
- **D. Design and optimisation challenge.** Design: list the checks a bank should run before debiting in a good order, and for each say what goes wrong if it is moved. Assume a per-payment limit of Rs 2000.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def fresh(L, payer=1000, payee=400):
    led = L.Ledger()
    led.deposit("payer", payer)
    led.deposit("payee", payee)
    return led


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    led = fresh(L)
    checks.append(("balances are sums of entries", led.balance("payer") == 1000 and led.balance("payee") == 400))
    checks.append(("the whole ledger sums to zero after deposits", led.ledger_sum() == 0 and led.customer_total() == 1400))

    # worked example: T1 pay 250, T1 again, T2 pay 900
    r1 = L.pay(led, "T1", "payer", "payee", 250)
    r2 = L.pay(led, "T1", "payer", "payee", 250)
    r3 = L.pay(led, "T2", "payer", "payee", 900)
    checks.append(("worked example: 250, retry of 250, then 900 gives payer 750, payee 650, total 1400",
                   r1["status"] == "ok" and r2["replayed"] and r3["reason"] == "insufficient balance"
                   and led.balance("payer") == 750 and led.balance("payee") == 650 and led.customer_total() == 1400))
    checks.append(("every transaction's entries sum to zero",
                   all(sum(a for t, _, a in led.entries if t == tid) == 0 for tid in {e[0] for e in led.entries})))
    checks.append(("the retry returns the same status as the first reply", r2["status"] == r1["status"] and r2["replayed"] and not r1["replayed"]))

    # atomicity
    a, c = fresh(L), fresh(L)
    for led2, fn in ((c, L.transfer_careless), (a, L.transfer_atomic)):
        try:
            fn(led2, "X", "payer", "payee", 250, crash=True)
        except L.Crash:
            pass
    checks.append(("atomic transfer: a crash leaves the ledger unchanged", a.customer_total() == 1400 and a.ledger_sum() == 0 and a.balance("payer") == 1000))
    checks.append(("careless transfer: a crash destroys 250 rupees", c.customer_total() == 1150 and c.ledger_sum() == -250))
    ok = fresh(L)
    L.transfer_atomic(ok, "Y", "payer", "payee", 250)
    checks.append(("atomic transfer without a crash moves exactly 250", ok.balance("payer") == 750 and ok.balance("payee") == 650))

    # order of checks
    good, bad = fresh(L), fresh(L)
    L.pay(good, "T1", "payer", "payee", 900)
    L.pay(bad, "T1", "payer", "payee", 900)
    again_good = L.pay(good, "T1", "payer", "payee", 900)
    again_bad = L.pay(bad, "T1", "payer", "payee", 900, dup_first=False)
    checks.append(("duplicate check first: retry of a 900 payment says ok (replayed)", again_good["status"] == "ok" and again_good["replayed"]))
    checks.append(("balance check first gives the wrong answer for a retry (the bug)", again_bad["status"] == "declined"))
    checks.append(("a retry never moves money twice", good.balance("payer") == 100 and good.balance("payee") == 1300))

    # each check on its own
    led = fresh(L)
    checks.append(("unauthenticated request is refused and nothing moves",
                   L.pay(led, "A1", "payer", "payee", 10, authenticated=False)["reason"] == "not authenticated" and led.balance("payer") == 1000))
    checks.append(("zero, negative and fractional amounts are refused",
                   [L.pay(led, f"B{i}", "payer", "payee", x)["reason"] for i, x in enumerate((0, -5, 2.5))] == ["bad amount"] * 3))
    checks.append(("over the per-payment limit is refused", L.pay(led, "C1", "payer", "payee", 2001)["reason"] == "over per-payment limit"))
    checks.append(("paying yourself is refused", L.pay(led, "D1", "payer", "payer", 10)["reason"] == "same account"))
    checks.append(("exactly the whole balance is allowed (boundary)", L.pay(led, "E1", "payer", "payee", 1000)["status"] == "ok" and led.balance("payer") == 0))
    checks.append(("one rupee over the balance is refused", L.pay(led, "E2", "payee", "payer", 1401)["reason"] == "insufficient balance"))
    checks.append(("no balance went negative", led.balance("payer") >= 0 and led.balance("payee") >= 0 and led.customer_total() == 1400))

    # retries
    checks.append(("3 retries of 250 with the id check: payer 750", L.retry_demo(250, 3, True) == (750, 650)))
    checks.append(("3 retries of 250 without it: payer 250, payee 1150, total still 1400", L.retry_demo(250, 3, False) == (250, 1150)))
    checks.append(("formula agrees with the real ledger for 1 to 6 retries",
                   all(L.retry_demo(250, k, False) == (L.payer_after(250, k, idempotent=False), 1400 - L.payer_after(250, k, idempotent=False)) for k in range(1, 7))))
    checks.append(("without the id check retries stop when the balance runs out (never negative)", L.payer_after(250, 8, idempotent=False) == 0))
    checks.append(("an amount above the balance is never charged", L.charge_count(1200, 3) == 0 and L.charge_count(1200, 3, idempotent=False) == 0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
